# Mae Na Rua + WMB Phayao — Daily Pipeline (Colab)

รันเรียงจากบนลงล่างทุกครั้งที่เปิด session ใหม่ (ห้ามกด "Runtime > Run all" ข้ามจากอันเก่า —
เซลล์ทดสอบ Phase 0-5 (ERA5T decode test, GEE auth test, MEI/CHIRPS เดี่ยว, SAR classification เต็ม)
ถูกลบออกจาก notebook นี้แล้ว เพราะเป็นการทดสอบระหว่างย้ายระบบที่ผ่านแล้ว ไม่ต้องรันซ้ำ — โค้ดเดิมยังอยู่ใน
`colab_migration/COLAB_MIGRATION_PLAN.md` ถ้าต้องย้อนดู

รายละเอียด/troubleshooting เต็มๆ ดูที่ `colab_migration/DAILY_RUNBOOK.md`

In [ ]:
#✅ ทุกวัน (ไม่ persist ข้าม session)
# Cell 1 — git clone ทั้ง 2 repo จาก GitHub ตรงๆ + bridge ไฟล์ใหญ่จาก Drive (2026-09-08 เปลี่ยน)
#
# เดิม: mount Drive แล้วชี้ path ตรงเข้า Drive-mounted mirror -- ต้องรอ Windows รัน sync_to_drive.bat
# ก่อนถึงจะสดจริง (ผู้ใช้ต้องเปิดโน๊ตบุ๊คทุกวัน)
# ใหม่: git clone ทั้ง 2 repo ตรงจาก GitHub ทุก session (repo public, clone/pull ไม่ต้องใช้ token) --
# ได้โค้ด + ไฟล์ที่ track ใน git สดที่สุดเสมอ ไม่ต้องพึ่ง Windows/Drive sync รายวันอีกต่อไป
#
# ยัง mount Drive อยู่ 1 จุด: ไฟล์ใหญ่เกิน 100MB ที่ .gitignore กันไว้ (rf_model/catboost model +
# WMB DEM/lulc/soil raster) -- ไฟล์พวกนี้ไม่เปลี่ยนรายวัน แค่ตอน retrain โมเดล/อัปเดตแผนที่ภูมิประเทศ
# นานๆ ครั้ง ใช้ path เดิมบน Drive เป๊ะ (DRIVE_BASE)

from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, shutil
from pathlib import Path

DRIVE_BASE = "/content/drive/MyDrive/Colab Notebooks/Mae_Na_Rua"   # ใช้แค่ bridge ไฟล์ใหญ่ที่ gitignore กันไว้เท่านั้น (ไม่ใช่แหล่งโค้ด/ข้อมูลหลักอีกต่อไป)

REPO_ROOT = "/content/repos"
PROJECT_WEB = f"{REPO_ROOT}/maenaruea-water-web"
PROJECT_WMB = f"{REPO_ROOT}/WMB_Phayao"
PIPELINE_DIR = f"{PROJECT_WEB}/01_data/scripts and code/pipeline"          # อ่านอย่างเดียว ห้ามเขียน
COLAB_MIGRATION_DIR = f"{PROJECT_WEB}/01_data/scripts and code/colab_migration"
WMB_COLAB_MIGRATION_DIR = f"{PROJECT_WMB}/colab_migration"

def _fresh_clone(repo_url, dest):
    if Path(dest).exists():
        shutil.rmtree(dest)
    subprocess.run(["git", "clone", repo_url, dest], check=True, capture_output=True, text=True)

os.makedirs(REPO_ROOT, exist_ok=True)
_fresh_clone("https://github.com/mpdox30/maenarua-water-web.git", PROJECT_WEB)
_fresh_clone("https://github.com/mpdox30/WMB_Phayao.git", PROJECT_WMB)
print("git clone เสร็จ: maenarua-water-web + WMB_Phayao")

# --- bridge ไฟล์ใหญ่ที่ .gitignore กันไว้ (rare-change) จาก Drive เข้า git checkout ---
_BRIDGE_FILES = [
    ("maenaruea-water-web/01_data/scripts and code/Water_demand/active/catboost_models.pkl",
     f"{PROJECT_WEB}/01_data/scripts and code/Water_demand/active/catboost_models.pkl"),
    ("maenaruea-water-web/01_data/scripts and code/Water_demand/active/rf_model_v3b_final.pkl",
     f"{PROJECT_WEB}/01_data/scripts and code/Water_demand/active/rf_model_v3b_final.pkl"),
]
_BRIDGE_DIRS = [
    ("WMB_Phayao/01_raw_data/DEM", f"{PROJECT_WMB}/01_raw_data/DEM"),
    ("WMB_Phayao/01_raw_data/lulc", f"{PROJECT_WMB}/01_raw_data/lulc"),
    ("WMB_Phayao/01_raw_data/soil_ldd", f"{PROJECT_WMB}/01_raw_data/soil_ldd"),
    ("WMB_Phayao/02_processed/dtm", f"{PROJECT_WMB}/02_processed/dtm"),
    ("WMB_Phayao/02_processed/flow_direction", f"{PROJECT_WMB}/02_processed/flow_direction"),
]
for _rel_src, _dst in _BRIDGE_FILES:
    _src = f"{DRIVE_BASE}/{_rel_src}"
    if os.path.exists(_src):
        os.makedirs(os.path.dirname(_dst), exist_ok=True)
        shutil.copy2(_src, _dst)
        print("bridge OK:", _dst)
    else:
        print("!! ไม่พบไฟล์ใหญ่บน Drive (เช็คว่ายังอยู่ path เดิมไหม):", _src)
for _rel_src, _dst in _BRIDGE_DIRS:
    _src = f"{DRIVE_BASE}/{_rel_src}"
    if os.path.exists(_src):
        os.makedirs(_dst, exist_ok=True)
        shutil.copytree(_src, _dst, dirs_exist_ok=True)
        print("bridge OK:", _dst)
    else:
        print("!! ไม่พบโฟลเดอร์ใหญ่บน Drive (เช็คว่ายังอยู่ path เดิมไหม):", _src)

for p in [PROJECT_WEB, PIPELINE_DIR, COLAB_MIGRATION_DIR, PROJECT_WMB, WMB_COLAB_MIGRATION_DIR]:
    print(p, "->", "OK" if os.path.exists(p) else "!! ไม่พบ ตรวจสอบ git clone")


In [2]:
#✅ ทุกวัน (ไม่ persist ข้าม session)
# ติดตั้ง dependency ของ ERA5T

!pip install -q cdsapi cfgrib eccodes ecmwflibs xarray
import eccodes
print("eccodes version:", eccodes.codes_get_api_version())

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.6/91.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 70.3 MB/s eta 0:00:00
eccodes version: 2.48.0


In [3]:
#✅ ทุกวัน (ไม่ persist ข้าม session)
# โหลด CDS credential จาก Colab Secret (CDSAPI_URL / CDSAPI_KEY) เป็น .cdsapirc

from google.colab import userdata

cdsapi_url = userdata.get('CDSAPI_URL')
cdsapi_key = userdata.get('CDSAPI_KEY')

with open('/root/.cdsapirc', 'w') as f:
    f.write(f"url: {cdsapi_url}\nkey: {cdsapi_key}\n")
print("เขียน .cdsapirc แล้ว")

เขียน .cdsapirc แล้ว


In [4]:
#✅ ทุกวัน (ไม่ persist ข้าม session)
# เพิ่ม path สำหรับ import โมดูลที่พอร์ตแล้ว/ต้นฉบับ

import sys
sys.path.insert(0, PIPELINE_DIR)
sys.path.insert(0, COLAB_MIGRATION_DIR)

In [5]:
#✅ ทุกวัน (ไม่ persist ข้าม session)
# ตั้ง GEE Service Account จาก Colab Secret

!pip install -q earthengine-api

from google.colab import userdata
import os

gee_sa_key_json = userdata.get('GEE_SA_KEY_JSON')
gee_sa_email = userdata.get('GEE_SA_EMAIL')

gee_key_path = "/content/gee_sa_key.json"
with open(gee_key_path, "w") as f:
    f.write(gee_sa_key_json)

os.environ["GEE_SERVICE_ACCOUNT_EMAIL"] = gee_sa_email
os.environ["GEE_SERVICE_ACCOUNT_KEY"] = gee_key_path

print("ตั้ง env var แล้วสำหรับ:", gee_sa_email)

ตั้ง env var แล้วสำหรับ: gee-service-account@maenaruea-water-pipeline.iam.gserviceaccount.com


In [6]:
#✅ ทุกวัน (ไม่ persist ข้าม session)
# dependency โมเดลทำนาย (Water Demand / Reservoir Inflow)
# 2026-07-22 เพิ่ม rasterio: chirps_feature.py มี tier ใหม่ "CHIRPS Prelim FTP" ที่อ่านไฟล์
# .tif ตรงจาก CHC FTP ด้วย rasterio (ดึงข้อมูลเร็วกว่า Prelim เดิม ~3 เท่า, lag ~7 วัน) --
# ถ้าไม่ติดตั้ง จะไม่ error แต่จะ silent fallback ไป tier Prelim เดิม (community, ช้ากว่า)

!pip install -q catboost==1.2.10 lightgbm==4.6.0 openpyxl==3.1.5 rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.3 MB/s eta 0:00:00


In [7]:
#✅ ทุกวัน (ไม่ persist ข้าม session)
# dependency ของ daily_update_colab.py (WMB_Phayao) — requests: โหลด gdrive_log, plotly: rebuild Reservoirs_inflow.html

!pip install -q requests openpyxl plotly

## รันประจำวันจากนี้ลงไป (ลำดับสำคัญ — WMB_Phayao ก่อน Mae Na Rua ก่อน push)

In [8]:
#✅ ทุกวัน
# รัน WMB_Phayao daily_update_colab.py (พยากรณ์น้ำท่วม 7 วัน + inflow อ่าง 5 อ่าง)
# ห้ามใส่ --offline (นั่นคือโหมดทดสอบ ไม่ดึงข้อมูลจริง)

import subprocess, os

r = subprocess.run(
    ["python3", f"{WMB_COLAB_MIGRATION_DIR}/daily_update_colab.py"],
    env={**os.environ, "WMB_ROOT": PROJECT_WMB},
    capture_output=True, text=True,
)
print(r.stdout)
if r.returncode != 0:
    print("STDERR:", r.stderr[-2000:])

﻿ดาวน์โหลดจาก Drive แล้ว (9747 KB)
gdrive_log: รวมเข้า store แล้ว (210049 แถวสะสม)
rain_res002: ถึง 2026-08-20 (408 วัน)
rain_wbyn: ถึง 2026-08-20 (34 วัน)
level_pyo001: ถึง 2026-08-20 (35 วัน)
ระดับกว๊าน: ต่อด้วย PYO001+0.369 อีก 35 วัน (ถึง 2026-08-20)
ฝน: ต่อด้วยสถานีจริงอีก 41 วัน (ถึง 2026-08-20)

วันฐานพยากรณ์ (มีทั้งฝน+ระดับ): 2026-08-20
apply alpha_C=1.258 ทุก catchment/reservoir (จาก C_uncalibrated) [candidate D]
res_maenarua_observed_daily.csv: อัปเดต 36 วัน จาก /content/drive/MyDrive/Colab Notebooks/Mae_Na_Rua/maenaruea-water-web/01_data/Reservoirs/inflow_auto/RES002_daily_computed.csv
inflow_forcing/release_forcing Res_MaeNaRua: inflow 419 วัน, release 419 วัน
inflow_forcing ใช้ข้อมูลจริงกี่วัน: {'Res_MaeNaRua': 409}
บันทึก: 09_live/output/Flood_output_forecast_latest.csv (126 แถว, 18 node, 2026-08-21–2026-08-27) -- input ให้ gen_ponding_raster_forecast.py

===== พยากรณ์ 7 วันข้างหน้า (วันฐาน 2026-08-20) =====
2026-08-21 กว๊าน 391.77 (391.73–391.83) ⚠️ กว๊าน>ตลิ่งร่องไฮ
202

In [9]:
import subprocess, os

r = subprocess.run(
    ["python3", "sar_background_job.py"],
    cwd=f"{PIPELINE_DIR}",
    capture_output=True, text=True,
)
print(r.stdout)
if r.returncode != 0:
    print("STDERR:", r.stderr[-2000:])

{
  "ran": false,
  "reason": "not_due_or_no_new_image",
  "result": null
}



In [11]:
#✅ ทุกวัน
# รัน Mae Na Rua หลัก — climate features (MEI/CHIRPS/ERA5T) -> อ่าน SAR จากแคช -> ทำนาย -> เขียน latest.json

import importlib
import data_pipeline_colab as dp
importlib.reload(dp)

result = dp.run_pipeline()
print("status:", result.status)
print("step_status:", result.step_status)
print("errors:", result.errors)

2026-08-20 02:09:41,823 [INFO] data_pipeline: Step 1/5: ดึงข้อมูลโทรมาตร
2026-08-20 02:09:42,497 [WARNING] data_pipeline: ใช้ mock data - ยังไม่ได้เชื่อม API จริง (TELEMETRY_API_URL is None)
2026-08-20 02:09:42,499 [INFO] data_pipeline: Step 2/5: ดึง MEI + CHIRPS + ERA5T (climate features)
2026-08-20 02:09:44,581 [INFO] data_pipeline: Downloading ONI (Nino 3.4 cross-check) from https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt ...
2026-08-20 02:09:45,712 [INFO] data_pipeline: Downloading MEI v2 from https://psl.noaa.gov/enso/mei/data/meiv2.data ...
2026-08-20 02:09:46,493 [INFO] data_pipeline: MEI ล่าสุดที่ดึงได้จาก NOAA คือช่วง 2026-07 (อายุ 20 วัน ไม่เกินเกณฑ์ 60 วัน) — ปกติ
2026-08-20 02:09:46,514 [INFO] data_pipeline: MEI feature พร้อมใช้: as_of=2026-W34, MEI=2.41, MEI_lag4=2.41, MEI_lag8=1.52 (latest_actual_period=2026-07, data_age_days=20, is_stale=False, stale_fallback_used=True, mei_reporting_lag_risk=False, nino34_oni_latest={'season': 'MJJ', 'year': 2026, 'anom': 1.39}

ede179b4b0fe4e4783e88a5e76c07f3b.grib:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/cfgrib/xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)
/usr/local/lib/python3.12/dist-packages/cfgrib/xarray_store.py:51: FutureWarning: In a future version of xarray the default value for compat will change from compat='no_conflicts' to compat='override'. This is likely to lead to different results when combining overlapping variables with the same name. To opt in to new defaults and get rid of these warnings now use `set_options(use_new_combine_kwarg_defaults=True) or set compat explicitly.
  o = xr.merge([o, ds], **kwargs)
/usr/local/lib/python3.12/dist-packages/cfgrib

[OK] เขียนผลลัพธ์ไปที่ /content/drive/MyDrive/Colab Notebooks/Mae_Na_Rua/maenaruea-water-web/01_data/scripts and code/colab_migration/era5t_output/era5t_week_2026-08-20.json


2026-08-20 02:12:36,791 [INFO] data_pipeline: โหลด Kc lookup จาก /content/drive/MyDrive/Colab Notebooks/Mae_Na_Rua/maenaruea-water-web/01_data/scripts and code/Water_demand/active/kc_weekly_lookup_all_crops.csv สำเร็จ (260 รายการ)
2026-08-20 02:12:36,792 [WARNING] data_pipeline: คำนวณสัดส่วน GIR_B ต่ออ่างไม่ได้สัปดาห์นี้ (ผลรวม FAO-56 ทุกอ่าง = 0.00 -- อาจเป็นสัปดาห์ที่ P_eff >= ETc ทุกพืชทุกอ่าง ไม่มี deficit ต้องเติมน้ำเลย) -- reservoir_breakdown จะไม่แสดงรอบนี้
2026-08-20 02:12:36,794 [INFO] data_pipeline: zone_B: ยังคำนวณสัดส่วน GIR ต่ออ่างไม่ได้รอบนี้ (SAR ยังไม่มี zone_b_reservoir_area_ha, หรือ climate สัปดาห์นี้ยังไม่พร้อม, หรือผลรวม FAO-56 เป็น 0) -- reservoir_breakdown ใน latest.json จะไม่มีรอบนี้
2026-08-20 02:12:36,818 [WARNING] data_pipeline: Climate prediction readiness (zone_A): fallback — climate data as_of สัปดาห์ 2026-W31 ไม่ใช่สัปดาห์ปัจจุบัน (2026-W34) เพราะสัปดาห์ปัจจุบันยังไม่มีข้อมูลครบ 7/7 วัน — ใช้สัปดาห์ล่าสุดที่ปิดแล้วและครบข้อมูลจริงแทน
2026-08-20 02:12:36,82

status: ok
step_status: {'telemetry': 'ok', 'climate_features': 'partial', 'climate_prediction_readiness': 'fallback', 'sar_classification': 'ok', 'prediction': 'ok', 'prediction_water_demand': 'ok', 'prediction_reservoir_inflow': 'ok', 'save_results': 'ok'}
errors: []


In [ ]:
#✅ ทุกวัน
# push ผลลัพธ์ขึ้น GitHub -- รวมเป็นรอบ pull+commit+push เดียว (2026-09-08 เปลี่ยน: เดิมแยก 2 เซลล์/
# 2 clone ต่างหาก (#7.5+#8.5 sync ml_features_live.csv กับ push_daily_data() 4 ไฟล์) ตอนนี้ PROJECT_WEB
# เอง (จาก Cell 1) เป็น git clone อยู่แล้ว เลยรวมเป็นรอบเดียวได้ ลด clone ซ้ำซ้อน)
#
# ไฟล์ที่ push: ml_features_live.csv (input พยากรณ์วันถัดไป), latest.json, flood_latest.json,
# reservoir_inflow.json, flood_depth_forecast.png -- ไฟล์อื่น (HTML/โค้ด) ผู้ใช้ push เองจาก Windows

from google.colab import userdata
import subprocess
from pathlib import Path
from datetime import datetime

GITHUB_PAT = userdata.get('GITHUB_PAT')
_repo_url_with_token = f"https://{GITHUB_PAT}@github.com/mpdox30/maenarua-water-web.git"
subprocess.run(["git", "-C", PROJECT_WEB, "remote", "set-url", "origin", _repo_url_with_token], check=True)
subprocess.run(["git", "-C", PROJECT_WEB, "config", "user.name", "Mae Na Rua Pipeline (Colab)"], check=True)
subprocess.run(["git", "-C", PROJECT_WEB, "config", "user.email", "mp.dox69@gmail.com"], check=True)

FILES_TO_PUSH = [
    "01_data/scripts and code/pipeline/ml_features_live.csv",
    "03_website/assets/data/latest.json",
    "03_website/assets/data/flood_latest.json",
    "03_website/assets/data/reservoir_inflow.json",
    "03_website/assets/data/flood_depth_forecast.png",
]

_existing = [f for f in FILES_TO_PUSH if (Path(PROJECT_WEB) / f).exists()]
_missing = [f for f in FILES_TO_PUSH if f not in _existing]
if _missing:
    print("ข้าม (ไม่พบไฟล์):", _missing)

if not _existing:
    print("ไม่มีไฟล์ให้ push เลย")
else:
    subprocess.run(["git", "-C", PROJECT_WEB, "add"] + _existing, check=True)
    _diff = subprocess.run(["git", "-C", PROJECT_WEB, "diff", "--cached", "--stat"], capture_output=True, text=True)
    if not _diff.stdout.strip():
        print("ข้อมูลไม่เปลี่ยนจากรอบก่อน -- ไม่ commit/push")
    else:
        _msg = f"Auto-update: pipeline data {datetime.now().strftime('%Y-%m-%d %H:%M')} (Colab)"
        subprocess.run(["git", "-C", PROJECT_WEB, "commit", "-m", _msg], check=True)

        _MAX_PUSH_RETRIES = 3
        for _attempt in range(1, _MAX_PUSH_RETRIES + 1):
            _result = subprocess.run(["git", "-C", PROJECT_WEB, "push", "origin", "HEAD:master"], capture_output=True, text=True)
            if _result.returncode == 0:
                print(f"push สำเร็จ (ลองครั้งที่ {_attempt}):", _msg)
                break
            if "fetch first" not in _result.stderr and "non-fast-forward" not in _result.stderr:
                print("push ไม่สำเร็จ (ไม่ใช่ non-fast-forward -- ต้องเช็คเอง):")
                print(_result.stderr[-800:])
                break
            print(f"[INFO] ครั้งที่ {_attempt}: non-fast-forward -- pull แล้วลองใหม่ ...")
            _pull = subprocess.run(["git", "-C", PROJECT_WEB, "pull", "--no-rebase", "--no-edit", "origin", "master"], capture_output=True, text=True)
            if _pull.returncode != 0 or "CONFLICT" in _pull.stdout or "CONFLICT" in _pull.stderr:
                print("[WARN] pull ไม่สำเร็จ หรือเจอ conflict เนื้อไฟล์จริง -- หยุด ไม่ auto-resolve เช็คที่:", PROJECT_WEB)
                print(_pull.stdout[-500:], _pull.stderr[-500:])
                break
        else:
            print(f"[WARN] push ไม่สำเร็จหลังลอง {_MAX_PUSH_RETRIES} ครั้ง -- รันเซลล์นี้ใหม่อีกรอบภายหลัง")
